# Stage 6 + 7 — Geometry and LSP MATLAB ↔ Python GPU Parity

Bu test mevcut gerçek dataset CSV'sindeki **tüm bankaları** kullanır.

Stage 6:

\[
(gNB,RIS,UE)\rightarrow
\{gNB\!\to\!RIS,\ RIS\!\to\!gNB,\ RIS\!\to\!UE,\ UE\!\to\!RIS,\ d_{BR},d_{RU}\}
\]

Stage 7:

\[
(\text{scenario},f_c,a,d)
\rightarrow
\text{LSP}
\]

Sekiz scenario etiketi:

- UMi LOS/NLOS
- UMa LOS/NLOS
- RMa LOS/NLOS
- Indoor-Office LOS/NLOS

LSP portu mevcut MATLAB formüllerini aynen kullanır; fiziksel bir yeniden yorumlama yapılmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, json
import pandas as pd
import numpy as np
import torch

ROOT = Path('/content/drive/MyDrive/MyDrive/RIS')

MODULE = ROOT/'ris_gpu_geometry_lsp_stage67.py'
if not MODULE.exists():
    MODULE = Path('/content/ris_gpu_geometry_lsp_stage67.py')

assert MODULE.exists(), f"Module bulunamadı: {MODULE}"

sys.path.insert(0,str(MODULE.parent))

from ris_gpu_geometry_lsp_stage67 import (
    compare_stage67_golden_csv,
    benchmark_geometry_lsp,
)

print("CUDA:",torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU :",torch.cuda.get_device_name(0))
print("Loaded:",MODULE)

## MATLAB golden export

MATLAB'da:

```matlab
export_stage67_geometry_lsp_suite
```

çalıştır.

Default olarak:

```text
generate_train_scenarios_data_2000.csv
```

dosyasındaki tüm satırları kullanır ve:

```text
stage67_geometry_lsp_golden.csv
```

üretir.

Bu CSV'yi Colab `/content` altına yükle.

In [ ]:
GOLDEN = Path('/content/stage67_geometry_lsp_golden.csv')
assert GOLDEN.exists(), (
    "stage67_geometry_lsp_golden.csv dosyasını /content altına yükle."
)

df = pd.read_csv(GOLDEN)
print("Rows:",len(df))
print("BR scenarios:",sorted(df['scenario_BR'].unique()))
print("RU scenarios:",sorted(df['scenario_RU'].unique()))
display(df.head())

In [ ]:
# DOUBLE / FLOAT64 PARITY

device = 'cuda' if torch.cuda.is_available() else 'cpu'

m64 = compare_stage67_golden_csv(
    str(GOLDEN),
    device=device,
    parity=True,
)

rel64 = {k:v for k,v in m64.items() if k.endswith('_relFro')}
exact64 = {k:v for k,v in m64.items() if k.endswith('_exact')}

worst64_key = max(rel64,key=rel64.get)
worst64 = rel64[worst64_key]

print("Worst double metric:",worst64_key,worst64)
print("All exact fields:",min(exact64.values()))

assert worst64 < 1e-12, (
    f"Stage 6/7 double parity failed: {worst64_key}={worst64:.3e}"
)
assert min(exact64.values()) == 1.0

print("PASS: Stage 6 + 7 double MATLAB parity")

In [ ]:
# FLOAT32 / COMPLEX64-equivalent production sanity
# (these stages are real-valued, so float32 only)

m32 = compare_stage67_golden_csv(
    str(GOLDEN),
    device=device,
    parity=False,
)

rel32 = {k:v for k,v in m32.items() if k.endswith('_relFro')}
exact32 = {k:v for k,v in m32.items() if k.endswith('_exact')}

worst32_key = max(rel32,key=rel32.get)
worst32 = rel32[worst32_key]

print("Worst float32 metric:",worst32_key,worst32)
print("All exact fields:",min(exact32.values()))

assert worst32 < 1e-5, (
    f"Stage 6/7 float32 sanity failed: {worst32_key}={worst32:.3e}"
)
assert min(exact32.values()) == 1.0

print("PASS: Stage 6 + 7 float32 production sanity")

In [ ]:
# Optional GPU throughput benchmark.
if torch.cuda.is_available():
    bench = benchmark_geometry_lsp(
        batch_size=1_000_000,
        repeats=10,
        device='cuda',
    )
    print(json.dumps(bench,indent=2))

## Bu aşama geçince

Python artık kendi başına:

\[
(gNB,RIS,UE,\text{scenario},f_c)
\]

girdilerinden geometry + LSP + linear K üretmiş olacak.

Bundan sonra Stage 1–5 ile birleştirip ilk **full deterministic Python environment** testine geçebiliriz.